In [0]:
%run ../../config/utils

In [0]:
import pyspark.sql.functions as f
import pandas as pd
import numpy as np
import mlflow
from mlflow.client import MlflowClient
from mlflow.models.signature import infer_signature
from datetime import datetime


mlflow.set_registry_uri('databricks-uc')
mlflow.autolog(disable=True)

In [0]:
run_date_str = dbutils.widgets.get('run_as_date')
current_year = int(run_date_str[:4])
gm_preprocessing_uri = f"models:/{gm_preprocessing_catalog}@$tenure_group"
gm_model_uri = f"models:/{gm_model_catalog}@$tenure_group"

In [0]:
latest_run_date = spark.table(gm_etl_output).filter((f.col('RUN_DATE') <= run_date_str) & (f.col('DATASET_CD') == 'inference')).agg(f.max('RUN_DATE').alias('max_dt')).first()['max_dt']
members = spark.table(gm_etl_output).filter((f.col('RUN_DATE') == latest_run_date) & (f.col('DATASET_CD') == 'inference')).drop('RUN_DATE','DATASET_CD')

In [0]:
data_full = pd.DataFrame()

for tenure_group in ['new', 'tenured']:
    data_preprocessing = mlflow.sklearn.load_model(gm_preprocessing_uri.replace('$tenure_group', tenure_group))
    model = mlflow.xgboost.load_model(gm_model_uri.replace('$tenure_group', tenure_group))

    if tenure_group == 'new':
        data_spark = members.filter(members.TENURE_GROUP.isin(['new']))
        data_pd = data_spark.toPandas()
    else:
        data_spark = members.filter((f.col('TENURE_GROUP').isNull()) | (~f.col('TENURE_GROUP').isin('new')))
        data_pd = data_spark.toPandas()

    data_pd.columns = data_pd.columns.str.upper()

    X_selected = data_preprocessing.transform(data_pd)
    score = model.predict_proba(X_selected)

    data_id = data_spark.select('MBRSHP_SID','LATEST_MBRSHP_NBR').toPandas()
    data_id['Probability']= score[:, 1]
    data_id['Decile'] = 10 - pd.qcut(data_id['Probability'].rank(method='first'), 10, labels = False)
    print(data_id.groupby('Decile').agg({'MBRSHP_SID':'nunique','Probability':'min'}))
    data_full = pd.concat([data_full, data_id], ignore_index=True)     

In [0]:
data_full['Decile'] = 10 - pd.qcut(data_full['Probability'].rank(method='first'), 10, labels = False)
# data_full.groupby('Decile').agg({'MBRSHP_SID':'nunique','Probability':'min'})

In [0]:
data_full.columns = ['MBRSHP_SID','mbrshp_nbr','score','decile']
data_full['score'] = data_full['score'].round(4)
data_full['score_date'] = datetime.strptime(run_date_str, '%Y-%m-%d').date()

In [0]:
data_full_spark = (
                    spark.createDataFrame(data_full)
                        .withColumn('MBRSHP_SID', f.col('MBRSHP_SID').cast('long'))
                        .withColumn('mbrshp_nbr', f.col('mbrshp_nbr').cast('long'))
                        .withColumn('score', f.col('score').cast('decimal(5,4)'))
                        .withColumn('decile', f.col('decile').cast('int'))
                        .withColumn('score_date', f.col('score_date').cast('date'))
)

data_full_spark.write.mode('overwrite').option('replaceWhere', f"score_date = '{run_date_str}'").saveAsTable(gm_scores)